In [9]:
###Data Injection
from langchain_core.documents import Document

In [10]:
doc = Document(
    page_content="this is the environment that make u more effective",
    metadata={
        "source":"the Book",
        "Author":"VNY Sambangi",
        "pages":3,
        "timestamp":"2026-08-04"
    }
)
doc

Document(metadata={'source': 'the Book', 'Author': 'VNY Sambangi', 'pages': 3, 'timestamp': '2026-08-04'}, page_content='this is the environment that make u more effective')

In [11]:
import os
os.makedirs("../data/text_files", exist_ok=True)

In [12]:
sample_texts = {"../data/text_files/lattice.txt":"""Unblock work and unlock potential with your personal AI Agent Support employees, managers, and leaders alike in doing their best work by proactively surfacing insights, answering questions about policies or career growth, and reinforcing positive habits.""",
                "../data/text_files/chuncking.txt":"""There are two big reasons why chunking is necessary for any application involving vector databases or LLMs: to ensure embedding models can fit the data into their context windows, and to ensure the chunks themselves contain the information necessary for search.
All embedding models have context windows, which determine the amount of information in tokens that can be processed into a single fixed size vector. Exceeding this context window may means the excess tokens are truncated, or thrown away, before being processed into a vector. This is potentially harmful
as important context could be removed from the representation of the text, which prevents it from being surfaced during a search."""}
for filepath,content in sample_texts.items():
    with open(filepath, "w", encoding='utf-8')as f:
        f.write(content)
print("sample text files were created")


sample text files were created


In [14]:
from langchain_community.document_loaders import TextLoader

tl=TextLoader("../data/text_files/chuncking.txt", encoding='utf-8')
document =tl.load()
print(document)

[Document(metadata={'source': '../data/text_files/chuncking.txt'}, page_content='There are two big reasons why chunking is necessary for any application involving vector databases or LLMs: to ensure embedding models can fit the data into their context windows, and to ensure the chunks themselves contain the information necessary for search.\nAll embedding models have context windows, which determine the amount of information in tokens that can be processed into a single fixed size vector. Exceeding this context window may means the excess tokens are truncated, or thrown away, before being processed into a vector. This is potentially harmful\nas important context could be removed from the representation of the text, which prevents it from being surfaced during a search.')]


In [45]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader,PyMuPDFLoader
text_loader = DirectoryLoader(
    "../data/text_files",
    glob=["**/*.txt"],
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'},
    show_progress=False
)
pdf_loader=DirectoryLoader(
    "../data/pdf",
        glob=["**/*.pdf"],
        loader_cls=PyMuPDFLoader,
        show_progress=False
)
chuncks=text_loader.load() + pdf_loader.load() 
# for i in documents:
#     print(i.page_content)
chuncks

[Document(metadata={'source': '..\\data\\text_files\\chuncking.txt'}, page_content='There are two big reasons why chunking is necessary for any application involving vector databases or LLMs: to ensure embedding models can fit the data into their context windows, and to ensure the chunks themselves contain the information necessary for search.\nAll embedding models have context windows, which determine the amount of information in tokens that can be processed into a single fixed size vector. Exceeding this context window may means the excess tokens are truncated, or thrown away, before being processed into a vector. This is potentially harmful\nas important context could be removed from the representation of the text, which prevents it from being surfaced during a search.'),
 Document(metadata={'source': '..\\data\\text_files\\lattice.txt'}, page_content='Unblock work and unlock potential with your personal AI Agent Support employees, managers, and leaders alike in doing their best wor

In [46]:
import numpy as np 
import os
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [47]:
class EmbeddigManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. EMbedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]):
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape : {embeddings.shape}")
        return embeddings
    
    #initialize the embedding manager
embedding_manager = EmbeddigManager()
embedding_manager 

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1622.60it/s]


Model loaded successfully. EMbedding dimension: 384


C:\Users\vinay\AppData\Local\Temp\ipykernel_41536\83842482.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. EMbedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [49]:
class VectorStore:
    def __init__(self, collection_name : str = "Kamakshi", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata= {"description":"PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store {e}")
            raise
        
    def add_documents(self, documents: List[Any], embeddings:np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)} documents to vector store...")
        
        ids = []
        metadatas = []
        document_text = []
        embeddings_list = []
        
        for i,(doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            document_text.append(doc.page_content)
            
            embeddings_list.append(embedding.tolist())
            
        try:
            self.collection.add(
                ids=ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = document_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"total documents in collection : {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
        
vector_store = VectorStore()
VectorStore

Vector store initialized Collection: Kamakshi
Existing documents in collection: 1010


__main__.VectorStore

In [50]:
texts = [doc.page_content for doc in chuncks]
texts

['There are two big reasons why chunking is necessary for any application involving vector databases or LLMs: to ensure embedding models can fit the data into their context windows, and to ensure the chunks themselves contain the information necessary for search.\nAll embedding models have context windows, which determine the amount of information in tokens that can be processed into a single fixed size vector. Exceeding this context window may means the excess tokens are truncated, or thrown away, before being processed into a vector. This is potentially harmful\nas important context could be removed from the representation of the text, which prevents it from being surfaced during a search.',
 'Unblock work and unlock potential with your personal AI Agent Support employees, managers, and leaders alike in doing their best work by proactively surfacing insights, answering questions about policies or career growth, and reinforcing positive habits.',
 'D-ILA® Projector \n \n \nDLA-NZ700, 

In [51]:
#Convert the text to embeddings
texts = [doc.page_content for doc in chuncks]

#generate the embeddings
embeddings = (embedding_manager.generate_embeddings(texts))
vector_store.add_documents(chuncks, embeddings)

Generating embeddings for 1757 texts...


Batches: 100%|██████████| 55/55 [01:29<00:00,  1.62s/it]


Generated embeddings with shape : (1757, 384)
Adding 1757 documents to vector store...
Successfully added 1757 documents to vector store
total documents in collection : 2767


In [52]:
import os
print(os.getcwd())

c:\Users\vinay\OneDrive\Desktop\projectK\RAG\SimpleRAG\notebook


In [53]:
class RAGRetriever:
    def __init__(self, vector_store : VectorStore, embedding_manager : EmbeddigManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        
    def retrive(self, query: str, top_k:int = 5, score_threshold:float=0.0) -> List[Dict[str,Any]]:
        print(f"Retrive documents for query: '{query}'")
        print(f"Top k: {top_k}, Score threshold: {score_threshold}")
        
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrived_docs = []
            print(results.keys())
            print(results)
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i,(docid, document, metadata, distance) in enumerate (zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrived_docs.append({
                            'id' : docid,
                            'content' : document,
                            'metadata' : metadata,
                            'similarity_score' : similarity_score,
                            'distance' : distance,
                            'rank' : i+1
                        })
                print(f"Retrived {len(retrived_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrived_docs
        except Exception as e:
            print(f"Error during retrival : {e}")
            return[]
        
rag_retriever= RAGRetriever(vector_store, embedding_manager)

RAG Retrival

In [55]:
rag_retriever.retrive("Could u explain me the http onCOnnect")

Retrive documents for query: 'Could u explain me the http onCOnnect'
Top k: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.72it/s]

Generated embeddings with shape : (1, 384)
dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])
{'ids': [['doc_1ec9323a_1404', 'doc_bd7d12ed_1405', 'doc_bec55a5c_1462', 'doc_e5539f4b_1410', 'doc_be9bfc61_359']], 'embeddings': None, 'documents': [['XP Driver Developer’s Guide, Release Runtime v25\n• channel (integer()) – Channel data was received on. The channel should\nbe used in the Write() function when responding.\n• data (string()) – Data received from port.\n• handle (integer()) – (optional) Used to match Handle property of ob-\nject.\nReturns none –\nSample:\nfunction OnCommRX(channel, data, [instance], [handle])\n{\n...\n}\n7.7 HTTP Object\nThe HTTP object is used to communicate with devices using a network TCP connection. The driver\nshould allocate an instance of the HTTP object for each desired usage. The HTTP object is similar to\nthe TCP object with a few behavioral changes. The HTTP object does not attempt to keep the TCP\ncon

[{'id': 'doc_1ec9323a_1404',
  'content': 'XP Driver Developer’s Guide, Release Runtime v25\n• channel (integer()) – Channel data was received on. The channel should\nbe used in the Write() function when responding.\n• data (string()) – Data received from port.\n• handle (integer()) – (optional) Used to match Handle property of ob-\nject.\nReturns none –\nSample:\nfunction OnCommRX(channel, data, [instance], [handle])\n{\n...\n}\n7.7 HTTP Object\nThe HTTP object is used to communicate with devices using a network TCP connection. The driver\nshould allocate an instance of the HTTP object for each desired usage. The HTTP object is similar to\nthe TCP object with a few behavioral changes. The HTTP object does not attempt to keep the TCP\nconnection open. It is the responsibility of the user to call the Open method to keep the connection\ngoing. The HTTP object is also built on a lightweight socket engine that allows many connections with\nhigh performance and error detection. The HTTP obj

Integration vector DB with the context pipeline with llm output

In [ ]:
from openai import OpenAI, RateLimitError, APIError
# from clients import openai_client, CHAT_MODEL
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv()

openaiAPIkey = os.getenv("OPENAI_API_KEY")
llm = ChatOpenAI(api_key=openaiAPIkey, temperature=0.1, model='gpt-4o-mini', max_completion_tokens=1024)

#Rag function
def simpleRag(query, retriver, llm, top_k=5):
    results = rag_retriever.retriver(query, top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    prompt =  f"""You are a technical assistant helping RTI driver developers understand OEM/vendor API documentation for the device or system they are currently integrating.
                You will be given:              
                1. A developer's query              
                2. A set of retrieved documentation chunks (fetched via similarity search from a vector database of previously indexed API documentation)           
                RULES:          
                1. Answer ONLY using information contained in the provided context chunks below. Do not use prior knowledge of this API, this vendor, or similar APIs to fill gaps.       
                2. If the retrieved context does not contain enough information to answer the query, say so explicitly — do not guess, infer undocumented behavior, or hallucinate endpoints, parameters, request/response formats, error codes, or command syntax.           
                3. When you answer, cite which chunk(s) the information came from (e.g. "[Chunk 2]") so the developer can trace it back to the source document.          
                4. If multiple chunks contain conflicting or overlapping information (e.g. different versions of the same endpoint), point out the conflict rather than silently picking one.         
                5. Preserve exact technical details verbatim where precision matters — request/response JSON structures, command strings, hex/byte payloads, status/error codes, units, parameter names, and casing. Do not paraphrase or "clean up" these details in a way that changes them.     
                6. If the query is about implementation (e.g. "how do I structure the driver command for X"), answer at the level of what the documentation specifies (endpoints, payload structure, auth, polling/subscription model) — do not invent driver code unless the retrieved context includes actual reference driver code.           
                7. Keep responses concise and structured (use headers/bullets/code blocks) — developers are using this to quickly look up integration details, not read prose.      
                Context chunks:          
                {context}             
                Developer query:               
                {query}""" 
    response = llm.invoke([{"role": "system", "content": "You are a helpful assistant."},
                           {"role": "user", "content": prompt}])
    return response.content

In [ ]:
answer = simpleRag("How grouping is done with the denon device",rag_retriever,llm)
print(answer)

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("OPENAI_API_KEY"))
print(os.getenv("LANGSMITH_API_KEY"))